# Демо проектных упражнений модуля 9 — Фильтры корпуса и механизм распределения состояния

**Демо-реализация, не готовое решение полномасштабного проекта.** Демо 1 выполняет фильтры и точное сравнение шинглов на синтетических документах; метки качества заданы вручную. Демо 2 воспроизводит обмен и обновление частей матрицы на CPU. Это подготовка к проектам, без MinHash, классификатора, настоящего FSDP и измерений GPU.
Все входы малы, детерминированы и работают без сети на CPU. Цель — проверить
интерфейсы, метрики, инварианты и структуру отчёта до расходования GPU/API-бюджета.

In [ ]:
from __future__ import annotations
import importlib.util, subprocess, sys
for package in ("numpy", "matplotlib"):
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])


import math
import random
import time
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import numpy as np

SEED = 20260801
random.seed(SEED)
np.random.seed(SEED)
print(f"воспроизводимый тестовый запуск; seed={SEED}")
DEMO_LANGUAGE = 'ru'

plt.rcParams.update({"font.family":"DejaVu Sans", "font.size":12, "axes.titlesize":15, "axes.labelsize":13, "xtick.labelsize":11, "ytick.labelsize":11, "figure.dpi":150})


## Демо 1. Почему документ покидает корпус

Восемь синтетических документов содержат точный дубликат после нормализации регистра, близкую копию, повторяющийся текст и искусственный адрес почты. Метки языка и качества уже заданы. Фильтры идут по порядку, а причина отсева — первая сработавшая; поэтому число в колонке зависит и от самого правила, и от предыдущих стадий. Дедупликация сравнивает документ только с ранее сохранёнными экземплярами.

In [ ]:
import re
from collections import Counter
# Synthetic documents; language and quality labels are supplied, not inferred.
docs=[
 ("a","en",.9,"matrix updates use gradients to change model weights"),
 ("b","en",.8,"MATRIX updates use gradients to change model weights"),
 ("c","en",.85,"matrix updates use gradients to change neural weights"),
 ("d","en",.1,"buy buy buy buy buy buy buy buy"),
 ("e","ru",.9,"Градиенты помогают обновлять параметры модели при обучении"),
 ("f","xx",.8,"unknown script placeholder text for this example"),
 ("g","en",.8,"Email example.person@example.test for the synthetic exercise"),
 ("h","en",.9,"attention reads values by comparing queries and keys"),
]
def words(text): return re.findall(r"\w+",text.casefold())
def shingles(text,k=2):
 w=words(text);return {tuple(w[i:i+k]) for i in range(len(w)-k+1)}
def jaccard(a,b): return len(a&b)/len(a|b) if a|b else 1.
def curate(order):
 kept=[];counts=Counter();seen=[]
 for docid,lang,quality,text in docs:
  failed=None
  for stage in order:
   if stage=="language" and lang not in {"en","ru"}:failed=stage
   elif stage=="quality" and quality<.5:failed=stage
   elif stage=="pii" and re.search(r"\S+@\S+",text):failed=stage
   elif stage=="dedup" and any(jaccard(shingles(text),s)>=.5 for s in seen):failed=stage
   if failed:break
  if failed:counts[failed]+=1
  else:kept.append(docid);seen.append(shingles(text))
 counts["kept"]=len(kept)
 assert sum(counts.values())==len(docs)
 return kept,counts
kept,counts=curate(["language","quality","pii","dedup"])
assert kept==["a","e","h"]
print("survival:",counts,"retained:",kept)
print("near-duplicate Jaccard:",jaccard(shingles(docs[0][3]),shingles(docs[2][3])))
# Cross-language paraphrases are not detected by this lexical mechanism.
assert jaccard(shingles(docs[0][3]),shingles(docs[4][3])) == 0
fig,ax=plt.subplots(figsize=(11,4.5),layout="constrained")
ax.bar(list(counts),list(counts.values()),color="#5E81AC")
ax.set_ylabel("Документы" if DEMO_LANGUAGE=="ru" else "Documents")
ax.set_title("Исключающие причины отсева" if DEMO_LANGUAGE=="ru" else "Exclusive retention/rejection counts")
plt.show()

В исходном порядке остаются a, e и h. Сумма исключающих причин и сохранённых документов равна восьми. Лексическое сравнение не распознаёт перевод или смысловой пересказ, а заданная метка качества не проверяет работу классификатора. Перестановка правил может изменить распределение причин, даже если конечный набор останется прежним.

## Демо 2. Принадлежащие процессу части и временно собранные веса

Разделим матрицу 8×4 по строкам между двумя условными процессами. На каждом шаге соберём полную матрицу, вычислим градиенты на двух равных частях данных, усредним их и разделим обратно по владельцам. Эталон обновляет ту же матрицу целиком. После проверки механизма перейдём к отдельной таблице памяти условной модели 1B: её схема AdamW не совпадает с float64-массивами опыта.

In [ ]:
# Simulate two ranks on CPU: all-gather weights, reduce-scatter gradients,
# update owned shards. No actual network, FSDP runtime, or GPU memory benchmark.
rng=np.random.default_rng(SEED)
x=rng.normal(size=(12,8));y=rng.normal(size=(12,4));w=rng.normal(size=(8,4))
shards=[a.copy() for a in np.array_split(w,2,axis=0)]
lr=.03
for step in range(4):
 full=np.concatenate(shards,axis=0)
 # Equal local batch sizes; average local gradients gives the global gradient.
 local=[]
 for xb,yb in zip(np.array_split(x,2),np.array_split(y,2)):
  local.append(xb.T@(xb@full-yb)/len(xb))
 grad=sum(local)/2
 refgrad=x.T@(x@w-y)/len(x)
 np.testing.assert_allclose(grad,refgrad,atol=1e-12)
 for i,g in enumerate(np.array_split(grad,2,axis=0)):shards[i]-=lr*g
 w-=lr*refgrad
 np.testing.assert_allclose(np.concatenate(shards),w,atol=1e-12)
assert all(s.nbytes==w.nbytes//2 for s in shards)
print("CPU shard parity: PASS; owned weight bytes/rank:",shards[0].nbytes,
      "temporary gathered weights:",full.nbytes)
# Different ledger: hypothetical mixed-precision AdamW, not this NumPy dtype.
P=1_000_000_000
for world in (1,2,8):
 for stage in (0,1,2,3):
  # Per parameter: weights 2 B, gradients 2 B, master+moments 12 B.
  per_parameter=(16,4+12/world,2+14/world,16/world)[stage]
  payload=P*per_parameter
  print(f"ZeRO-{stage}, {world} ranks: {payload/1e9:.3f} GB/rank")
 # An illustrative additional gather window and live activation reserve.
 gather=100_000_000*2 if world>1 else 0
 activations=500_000_000
 with_buffers=P*16/world+gather+activations
 print(f"ZeRO-3 plus declared buffers: {with_buffers/1e9:.3f} GB/rank")
print("Activations, gathers, communication buffers, and allocator reserve are additional.")

Совпадение четырёх шагов подтверждает усреднение градиентов и обновление соответствующих частей. Каждый владелец хранит половину весов, но сборка временно создаёт полную матрицу. В отдельной таблице стадии ZeRO делят разные составляющие; добавленные 0.2 GB окна сборки и 0.5 GB активаций не делятся на число процессов. Это заданные резервы, а не измеренный пик.

## Переход к полному проекту

Для У4 измените порядок правил и пороги и объясните каждый изменившийся результат. В необязательном опыте с внешними данными зафиксируйте ревизию, лицензию и размер выборки. Для У5 измените число частей и резерв буферов, сохранив равные локальные пакеты для простого усреднения; при неравных пакетах выведите веса среднего по числу примеров. Сетевое распределённое исполнение остаётся за границами задания.

### Минимальный отчёт

1. Версии данных, модели, кода и seed.
2. Основная метрика, а также качество, стоимость, задержка и память там, где это применимо.
3. Базовая линия, одинаковый бюджет и хотя бы один отрицательный контроль.
4. Неудачные прогоны и границы обоснованных выводов.